# Práctica 6: Manuel Alejandro Serrano Macias
# Modelo de Lenguaje Básico

In [204]:
def get_texts_from_file(path_corpus, path_truth):
  tr_txt = []
  tr_y = []

  with open(path_corpus, "r", encoding="utf-8") as f_corpus, open (path_truth, "r", encoding="utf-8") as f_truth:

    for twitt in f_corpus:
      tr_txt += [twitt]
    
    for label in f_truth:
      tr_y += [label]

  return tr_txt, tr_y

In [205]:
tr_txt, tr_y = get_texts_from_file("./mex20_train.txt","./mex20_train_labels.txt")

## Fase 1: Tratamiento y preprocesamiento de los datos a ser consumidos por un Lenguaje Model

In [206]:
from nltk.probability import FreqDist

class NgramData:
  
  def __init__(self, vocab_max, tokenizer, n=3):
    self.vocab_max = vocab_max
    self.n = n # orden del n-grama (1=unigrama ... 5=5-grama)
    self.tokenizer = tokenizer
    self.final_vocabulary = set()
    self.SOS = "<s>"
    self.EOS = "</s>"
    self.UNK = "<unk>"

  def fit(self, raw_texts):

    freq_dist = FreqDist()
    tokenized_corpus = [] # lista de listas de los tuits tokenizados

    for txt in raw_texts:
      tokens = self.tokenizer.tokenize(txt.lower())
      tokenized_corpus.append(tokens)
      for w in tokens:
        freq_dist[w] += 1

    self.final_vocabulary = {tok for tok, _ in freq_dist.most_common(self.vocab_max)} # lo más frecuente
    self.final_vocabulary.update([self.SOS, self.EOS, self.UNK]) # agregar tokens especiales

    transformed_corpus = []

    for tokens in tokenized_corpus:
      transformed_corpus.append(self.transform(tokens))

    return transformed_corpus

  def mask_oov(self, word):
    return self.UNK if word not in self.final_vocabulary else word

  def add_sos_eos(self, tokens):
    return [self.SOS] * (self.n - 1) + tokens + [self.EOS] # n-1 tokens <s> segun el orden
  
  def transform(self, tokens):
    transformed = []
    for w in tokens:
      transformed.append(self.mask_oov(w)) # out of vocabulary word
    transformed = self.add_sos_eos(transformed)
    return transformed

In [207]:
from nltk.tokenize import TweetTokenizer
tokenizer = TweetTokenizer()

trigram_data = NgramData(vocab_max=10000, tokenizer=tokenizer)

transformed_corpus = trigram_data.fit(tr_txt)
final_vocab = trigram_data.final_vocabulary

In [208]:
transformed_corpus

[['<s>',
  '<s>',
  '@usuario',
  '@usuario',
  '@usuario',
  'q',
  'se',
  'puede',
  'esperar',
  'del',
  'maricon',
  'de',
  'closet',
  'de',
  'la',
  'yañez',
  'aun',
  'recuerdo',
  'esa',
  'ves',
  'q',
  'lo',
  'vi',
  'en',
  'zona',
  'rosa',
  'viendo',
  'quien',
  'lo',
  'levantada',
  '</s>'],
 ['<s>',
  '<s>',
  '@usuario',
  'la',
  'piel',
  'nueva',
  'siempre',
  'arde',
  'un',
  'poquito',
  'los',
  'primeros',
  'días',
  '...',
  'y',
  'más',
  'con',
  'este',
  'puto',
  'clima',
  '</s>'],
 ['<s>',
  '<s>',
  'ustedes',
  'no',
  'se',
  'enamoran',
  'de',
  'mí',
  '…',
  'por',
  'tontas',
  '.',
  '</s>'],
 ['<s>',
  '<s>',
  'me',
  'las',
  'va',
  'a',
  'pagar',
  'esa',
  'puta',
  'gorda',
  'roba',
  'tuits',
  '...',
  '</s>'],
 ['<s>',
  '<s>',
  '@usuario',
  'la',
  'gente',
  'es',
  'tonta',
  'porque',
  'no',
  'se',
  'dan',
  'cuenta',
  'que',
  'tú',
  'haces',
  'a',
  'batman',
  'azul',
  '</s>'],
 ['<s>',
  '<s>',
  'estoy'

## Construir el N-gram Language Model

In [209]:
class NgramLanguageModel: 
    
  def __init__(self, lambda1=.3, lambda2=.3, lambda3=.4, lambda4=0, lambda5=0):
    # La idea es hacer TRES modelos de Lenguaje 
    # Esto de abajo me dice que tanto caso le hago a cada uno
    # NOTA: los lambdas siguen la numeracion de Jurafsky & Martin:
    # lambda1 = unigrama, lambda2 = bigrama, lambda3 = trigrama, lambda4 = 4-grama, lambda5 = 5-grama
    # (antes lambda1 era el trigrama y lambda3 el unigrama). Todos deben sumar 1.
    # Cambie el nombre de la clase por Ngram, ya que agregamos 4 y 5 grams.
    self.lambda1 = lambda1
    self.lambda2 = lambda2
    self.lambda3 = lambda3
    self.lambda4 = lambda4
    self.lambda5 = lambda5
    
    # A CONTAR:
    self.unigram_count = {} # aqui cuento unigramas (palabras)
    self.bigram_count = {} # aqui cuent bigramas (secuencias de tamaño 2)
    self.trigram_count = {} # aqui cuent trigramas (secuencias de tamaño 3)
    self.fourgram_count = {} # aqui cuento 4-gramas (secuencias de tamaño 4)
    self.fivegram_count = {} # aqui cuento 5-gramas (secuencias de tamaño 5)
    
    self.vocab = set()
    self.V = 0

  def train(self, transformed_corpus, final_vocab):
    self.vocab = final_vocab
    self.V = len(final_vocab)

    for tokens in transformed_corpus: # primero recorro tuit por tuit
      for i, w in enumerate(tokens): # a cada tuit le recorro las palabras
        # unigrama
        self.unigram_count[w] = self.unigram_count.get(w, 0) + 1

        # bigrama
        if i > 0:
          w_prev = tokens[i - 1] # la palabra previa
          self.bigram_count[(w_prev, w)] = self.bigram_count.get((w_prev, w), 0) + 1

        #trigramas
        if i > 1: #Solo si ya vi DOS token
          w_prev2 = tokens[i-2] # la palabra previa a la previa
          self.trigram_count[(w_prev2, w_prev, w)] =\
          self.trigram_count.get((w_prev2, w_prev, w), 0) + 1

        # 4-gramas
        if i > 2: # Solo si ya vi TRES tokens
          w_prev3 = tokens[i-3]
          self.fourgram_count[(w_prev3, w_prev2, w_prev, w)] =\
          self.fourgram_count.get((w_prev3, w_prev2, w_prev, w), 0) + 1

        # 5-gramas
        if i > 3: # Solo si ya vi CUATRO tokens
          w_prev4 = tokens[i-4]
          self.fivegram_count[(w_prev4, w_prev3, w_prev2, w_prev, w)] =\
          self.fivegram_count.get((w_prev4, w_prev3, w_prev2, w_prev, w), 0) + 1

    self.total_tokens = sum(self.unigram_count.values())

  def mask_oov(self,word):
    return "<unk>" if word not in self.vocab else word

  def unigram_probability(self, w): # P(w)
    numerator = self.unigram_count.get(self.mask_oov(w), 0) + 1
    denominator = self.total_tokens + self.V
    return numerator/denominator

  def bigram_probability(self, w_prev, w): # P(w|w_prev)
    w_prev = self.mask_oov(w_prev)
    w = self.mask_oov(w)
    numerator = self.bigram_count.get((w_prev, w),0) + 1
    denominator = self.unigram_count.get(w_prev, 0) + self.V
    return numerator/denominator

  def trigram_probability(self, w_prev2, w_prev, w): # P(w|w_prev2 w_prev)
    w_prev2 = self.mask_oov(w_prev2)
    w_prev = self.mask_oov(w_prev)
    w = self.mask_oov(w)
    numerator = self.trigram_count.get((w_prev2, w_prev, w),0) + 1
    denominator = self.bigram_count.get((w_prev2,w_prev), 0) + self.V
    return numerator/denominator

  def fourgram_probability(self, w_prev3, w_prev2, w_prev, w): # P(w|w_prev3 w_prev2 w_prev)
    w_prev3 = self.mask_oov(w_prev3)
    w_prev2 = self.mask_oov(w_prev2)
    w_prev = self.mask_oov(w_prev)
    w = self.mask_oov(w)
    numerator = self.fourgram_count.get((w_prev3, w_prev2, w_prev, w),0) + 1
    denominator = self.trigram_count.get((w_prev3, w_prev2, w_prev), 0) + self.V
    return numerator/denominator

  def fivegram_probability(self, w_prev4, w_prev3, w_prev2, w_prev, w): # P(w|w_prev4 w_prev3 w_prev2 w_prev)
    w_prev4 = self.mask_oov(w_prev4)
    w_prev3 = self.mask_oov(w_prev3)
    w_prev2 = self.mask_oov(w_prev2)
    w_prev = self.mask_oov(w_prev)
    w = self.mask_oov(w)
    numerator = self.fivegram_count.get((w_prev4, w_prev3, w_prev2, w_prev, w),0) + 1
    denominator = self.fourgram_count.get((w_prev4, w_prev3, w_prev2, w_prev), 0) + self.V
    return numerator/denominator

  def probability_of_word(self, w_prev2, w_prev, w): # P(w | w_prev2, w_prev)
    return self.lambda3 * self.trigram_probability(w_prev2, w_prev, w) + self.lambda2 * self.bigram_probability(w_prev, w) + self.lambda1 * self.unigram_probability(w) 

  def probability_of_word_4gram(self, w_prev3, w_prev2, w_prev, w): # P(w | w_prev3, w_prev2, w_prev)
    return self.lambda4 * self.fourgram_probability(w_prev3, w_prev2, w_prev, w) + self.lambda3 * self.trigram_probability(w_prev2, w_prev, w) + self.lambda2 * self.bigram_probability(w_prev, w) + self.lambda1 * self.unigram_probability(w)

  def probability_of_word_5gram(self, w_prev4, w_prev3, w_prev2, w_prev, w): # P(w | w_prev4, w_prev3, w_prev2, w_prev)
    return self.lambda5 * self.fivegram_probability(w_prev4, w_prev3, w_prev2, w_prev, w) + self.lambda4 * self.fourgram_probability(w_prev3, w_prev2, w_prev, w) + self.lambda3 * self.trigram_probability(w_prev2, w_prev, w) + self.lambda2 * self.bigram_probability(w_prev, w) + self.lambda1 * self.unigram_probability(w)

  def top_next_words(self, w_prev2, w_prev, top_k = 5, bottom_k = 5):
    candidates = []
    for cand in self.vocab:
      p = self.probability_of_word(w_prev2, w_prev, cand)
      candidates.append((cand, p))

    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:top_k]

  def top_next_words_4gram(self, w_prev3, w_prev2, w_prev, top_k = 5, bottom_k = 5):
    candidates = []
    for cand in self.vocab:
      p = self.probability_of_word_4gram(w_prev3, w_prev2, w_prev, cand)
      candidates.append((cand, p))

    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:top_k]

  def top_next_words_5gram(self, w_prev4, w_prev3, w_prev2, w_prev, top_k = 5, bottom_k = 5):
    candidates = []
    for cand in self.vocab:
      p = self.probability_of_word_5gram(w_prev4, w_prev3, w_prev2, w_prev, cand)
      candidates.append((cand, p))

    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:top_k]

  def sequence_probability(self, sequence): 
    # "Hola Mundo de NLP en la universidad Panamericana"
    # "Hola Mundo de"
    # "Mundo de NLP"
    # "de NLP en"

    import math
    log_prob = 0
    for i in range(2, len(sequence)):
      w_prev2 = sequence[i-2]
      w_prev = sequence[i-1]
      w = sequence[i]
      p = self.probability_of_word(w_prev2, w_prev, w)
      log_prob += math.log(p) # log_prob *= p

    return math.exp(log_prob)  

  def sequence_probability_4gram(self, sequence): 
    # "Hola Mundo de NLP en la universidad Panamericana"
    # "Hola Mundo de NLP"
    # "Mundo de NLP en"
    # "de NLP en la"

    import math
    log_prob = 0
    for i in range(3, len(sequence)):
      w_prev3 = sequence[i-3]
      w_prev2 = sequence[i-2]
      w_prev = sequence[i-1]
      w = sequence[i]
      p = self.probability_of_word_4gram(w_prev3, w_prev2, w_prev, w)
      log_prob += math.log(p) # log_prob *= p

    return math.exp(log_prob)  

  def sequence_probability_5gram(self, sequence): 
    # "Hola Mundo de NLP en la universidad Panamericana"
    # "Hola Mundo de NLP en"
    # "Mundo de NLP en la"
    # "de NLP en la universidad"

    import math
    log_prob = 0
    for i in range(4, len(sequence)):
      w_prev4 = sequence[i-4]
      w_prev3 = sequence[i-3]
      w_prev2 = sequence[i-2]
      w_prev = sequence[i-1]
      w = sequence[i]
      p = self.probability_of_word_5gram(w_prev4, w_prev3, w_prev2, w_prev, w)
      log_prob += math.log(p) # log_prob *= p

    return math.exp(log_prob)  

  def rank_permutations(self, tokens, top_k=5, bottom_k=5):
    from itertools import permutations 
    perms = set(permutations(tokens))
    print(len(perms))
    scores = [] 
    for p in perms: 
      prob = self.sequence_probability(list(p))
      scores.append((" ".join(p), prob))
    #print(scores)
    scores.sort(key=lambda x: x[1], reverse=True)
    best = scores[:top_k]
    worst = scores[-bottom_k:]
    #print(scores)
    return best, worst

  def rank_permutations_4gram(self, tokens, top_k=5, bottom_k=5):
    from itertools import permutations 
    perms = set(permutations(tokens))
    print(len(perms))
    scores = [] 
    for p in perms: 
      prob = self.sequence_probability_4gram(list(p))
      scores.append((" ".join(p), prob))
    scores.sort(key=lambda x: x[1], reverse=True)
    best = scores[:top_k]
    worst = scores[-bottom_k:]
    return best, worst

  def rank_permutations_5gram(self, tokens, top_k=5, bottom_k=5):
    from itertools import permutations 
    perms = set(permutations(tokens))
    print(len(perms))
    scores = [] 
    for p in perms: 
      prob = self.sequence_probability_5gram(list(p))
      scores.append((" ".join(p), prob))
    scores.sort(key=lambda x: x[1], reverse=True)
    best = scores[:top_k]
    worst = scores[-bottom_k:]
    return best, worst

  def generate_from_tokens_as_chatgpt(self, first_word, second_word, max_length = 25, delay = .1):
  
    if first_word not in self.vocab: 
      first_word = "<unk>"
    if second_word not in self.vocab: 
      second_word = "<unk>" 
  
    generated = [first_word, second_word] # aqui vamos a ir guardando el texto generado
    print(first_word, second_word, end=" ", flush=True)
  
    for _ in range(max_length):
      w_prev2 = generated[-2]
      w_prev = generated[-1]

      total_p = 0.0
      dist = []
      for w in self.vocab: # 10000
        p = self.probability_of_word(w_prev2, w_prev, w)
        dist.append((w,p))
        total_p += p


      # Diversidad del Modelo
      dist = sorted(dist, key=lambda x:x[1], reverse = True)[:3]
      
      total_p = sum(p for w, p in dist)
      dist = [(w, p/total_p) for w, p in dist]

      # Girar la ruleta
      import random, time, sys

      r = random.random()

      cumulative = 0.0
      chosen = None

      for w, p in dist:
        cumulative += p
        if r <= cumulative:
          chosen = w
          break

      if chosen is None or chosen == "</s>":
        break

      if chosen == "<s>":
        continue

      generated.append(chosen)
      print(chosen, end = " ", flush=True)
      time.sleep(delay)

    return generated

  def generate_from_tokens_as_chatgpt_4gram(self, first_word, second_word, third_word, max_length = 25, delay = .1):
  
    if first_word not in self.vocab: 
      first_word = "<unk>"
    if second_word not in self.vocab: 
      second_word = "<unk>"
    if third_word not in self.vocab: 
      third_word = "<unk>"
  
    generated = [first_word, second_word, third_word] # aqui vamos a ir guardando el texto generado
    print(first_word, second_word, third_word, end=" ", flush=True)
  
    for _ in range(max_length):
      w_prev3 = generated[-3]
      w_prev2 = generated[-2]
      w_prev = generated[-1]

      total_p = 0.0
      dist = []
      for w in self.vocab: # 10000
        p = self.probability_of_word_4gram(w_prev3, w_prev2, w_prev, w)
        dist.append((w,p))
        total_p += p


      # Diversidad del Modelo
      dist = sorted(dist, key=lambda x:x[1], reverse = True)[:3]
      
      total_p = sum(p for w, p in dist)
      dist = [(w, p/total_p) for w, p in dist]

      # Girar la ruleta
      import random, time, sys

      r = random.random()

      cumulative = 0.0
      chosen = None

      for w, p in dist:
        cumulative += p
        if r <= cumulative:
          chosen = w
          break

      if chosen is None or chosen == "</s>":
        break

      if chosen == "<s>":
        continue

      generated.append(chosen)
      print(chosen, end = " ", flush=True)
      time.sleep(delay)

    return generated

  def generate_from_tokens_as_chatgpt_5gram(self, first_word, second_word, third_word, fourth_word, max_length = 25, delay = .1):
  
    if first_word not in self.vocab: 
      first_word = "<unk>"
    if second_word not in self.vocab: 
      second_word = "<unk>"
    if third_word not in self.vocab: 
      third_word = "<unk>"
    if fourth_word not in self.vocab: 
      fourth_word = "<unk>"
  
    generated = [first_word, second_word, third_word, fourth_word] # aqui vamos a ir guardando el texto generado
    print(first_word, second_word, third_word, fourth_word, end=" ", flush=True)
  
    for _ in range(max_length):
      w_prev4 = generated[-4]
      w_prev3 = generated[-3]
      w_prev2 = generated[-2]
      w_prev = generated[-1]

      total_p = 0.0
      dist = []
      for w in self.vocab: # 10000
        p = self.probability_of_word_5gram(w_prev4, w_prev3, w_prev2, w_prev, w)
        dist.append((w,p))
        total_p += p


      # Diversidad del Modelo
      dist = sorted(dist, key=lambda x:x[1], reverse = True)[:3]
      
      total_p = sum(p for w, p in dist)
      dist = [(w, p/total_p) for w, p in dist]

      # Girar la ruleta
      import random, time, sys

      r = random.random()

      cumulative = 0.0
      chosen = None

      for w, p in dist:
        cumulative += p
        if r <= cumulative:
          chosen = w
          break

      if chosen is None or chosen == "</s>":
        break

      if chosen == "<s>":
        continue

      generated.append(chosen)
      print(chosen, end = " ", flush=True)
      time.sleep(delay)

    return generated

  def checar_probabilidad(self):
    print(sum(self.unigram_probability(w) for w in self.vocab))
    print(sum(self.bigram_probability("hijo",w) for w in self.vocab))
    print(sum(self.trigram_probability("hijo", "de", w) for w in self.vocab))
    print(sum(self.fourgram_probability("no", "me", "importa", w) for w in self.vocab))
    print(sum(self.fivegram_probability("no", "me", "importa", "lo", w) for w in self.vocab))

In [210]:
trigram_lm = NgramLanguageModel(lambda1=.01, lambda2=.29, lambda3=.7) # lambda1=unigrama, lambda3=trigrama
trigram_lm.train(transformed_corpus, final_vocab)

In [211]:
trigram_lm.checar_probabilidad()

1.0
0.9999999999999999
0.9999999999999999
1.0
1.0


In [212]:
trigram_lm.generate_from_tokens_as_chatgpt("vamonos", "a")

vamonos a mi me <unk> , <unk> <unk> , no se la verga , pero me vale verga . 

['vamonos',
 'a',
 'mi',
 'me',
 '<unk>',
 ',',
 '<unk>',
 '<unk>',
 ',',
 'no',
 'se',
 'la',
 'verga',
 ',',
 'pero',
 'me',
 'vale',
 'verga',
 '.']

In [213]:
trigram_lm.generate_from_tokens_as_chatgpt("<s>", "vete")

<s> vete 

['<s>', 'vete']

In [214]:
trigram_lm.generate_from_tokens_as_chatgpt("<s>", "hijo")

<s> hijo de la <unk> <unk> , <unk> <unk> 

['<s>', 'hijo', 'de', 'la', '<unk>', '<unk>', ',', '<unk>', '<unk>']

In [215]:
trigram_lm.generate_from_tokens_as_chatgpt("<s>", "de")

<s> de 

la verga , pero que me <unk> , <unk> , no se <unk> 

['<s>',
 'de',
 'la',
 'verga',
 ',',
 'pero',
 'que',
 'me',
 '<unk>',
 ',',
 '<unk>',
 ',',
 'no',
 'se',
 '<unk>']

In [216]:
trigram_lm.sequence_probability(["estamos", "en", "clase", "en", "la", "universidad"])

2.523678439174141e-14

In [217]:
trigram_lm.rank_permutations(["estamos", "en", "clase", "en", "la", "universidad"])

360


([('clase estamos en la universidad en', 4.456373783493826e-14),
  ('clase estamos en en la universidad', 4.3888315547325e-14),
  ('estamos clase en la universidad en', 3.8664663432335796e-14),
  ('estamos clase en en la universidad', 3.8080012429147796e-14),
  ('estamos en la universidad en clase', 2.524037962564892e-14)],
 [('en la en universidad estamos clase', 1.7931207652840133e-16),
  ('en la en estamos clase universidad', 1.7930920471675544e-16),
  ('en la estamos universidad clase en', 1.7897715886014985e-16),
  ('en la estamos clase universidad en', 1.7894585448150587e-16),
  ('en la estamos clase en universidad', 1.7377198848552506e-16)])

In [218]:
trigram_lm.rank_permutations(["sino", "gano", "me", "voy", "a", "la", "chingada"])

5040


([('sino gano me voy a la chingada', 1.1888548091707448e-14),
  ('gano sino me voy a la chingada', 1.1887446268415863e-14),
  ('gano me voy a la chingada sino', 5.397357006235533e-15),
  ('sino me voy a la chingada gano', 5.3664929083979225e-15),
  ('sino gano voy a la chingada me', 1.1961264465971988e-15)],
 [('a la gano me chingada voy sino', 2.1759882386650268e-20),
  ('a la gano chingada voy me sino', 2.1759832000981257e-20),
  ('a la gano me sino voy chingada', 2.1758702372755564e-20),
  ('a la sino voy chingada me gano', 2.1745073306434097e-20),
  ('a la gano voy chingada me sino', 2.1743866372593245e-20)])

In [219]:
trigram_lm.sequence_probability(["hijo", "de", "tu"])

0.0032612649062403625

In [220]:
trigram_lm.probability_of_word("hola", "mundo", "de")

0.0004074793480973208

In [221]:
trigram_lm.probability_of_word("vete", "a", "la")

0.008435733776723074

In [222]:
trigram_lm.top_next_words("hola", "mundo")

[('<s>', 0.0009549495876169024),
 ('</s>', 0.0006422652110836188),
 ('.', 0.0005984271954686746),
 (',', 0.00048766073766232456),
 ('de', 0.0004074793480973208)]

In [223]:
trigram_lm.top_next_words("hijo", "de")

[('la', 0.006861290280647399),
 ('<unk>', 0.00413394999513011),
 ('tu', 0.003261264906240363),
 ('su', 0.0021199707400528776),
 ('las', 0.0019805465121648332)]

In [224]:
trigram_lm.top_next_words("vamos", "a")

[('la', 0.007736761273210426),
 ('mi', 0.0023538458690032116),
 ('las', 0.0022625566452511635),
 ('su', 0.002243904467812189),
 ('los', 0.002007578103386118)]

In [225]:
trigram_lm.top_next_words("eres", "una")

[('<s>', 0.0009535225138758293),
 ('loca', 0.0008061544606125536),
 ('<unk>', 0.0007906828581326495),
 ('gorda', 0.0007587123964673913),
 ('de', 0.0007485687598868228)]

In [226]:
trigram_lm.trigram_probability("hola", "mundo", "de")

9.997000899730081e-05

In [227]:
trigram_lm.trigram_probability("vete", "a", "la")

0.0010983524712930605

In [228]:
trigram_lm.trigram_probability("hijo", "de", "la")

0.0003971800218449012

In [229]:
trigram_lm.trigram_probability("que", "cual", "como")

9.997000899730081e-05

In [230]:
trigram_lm.unigram_probability("hola")

9.731570837726056e-05

In [231]:
trigram_lm.bigram_probability("hola", "mundo")

9.986019572598362e-05

In [232]:
trigram_lm.bigram_probability("hijo", "de")

0.006836421282076687

In [233]:
trigram_lm.bigram_probability("vete", "a")

0.0012975346841002096

In [234]:
trigram_lm.bigram_probability("vamos", "a")

0.0015942606616181746

In [235]:
trigram_lm.bigram_probability("arriba", "computadora")

9.987016878058524e-05

In [236]:
trigram_lm.unigram_probability("madre")

0.0062687535479685344

In [237]:
trigram_lm.unigram_probability("universidad")

5.6767496553401994e-05

In [238]:
trigram_lm.unigram_probability("hijo")

0.000737977455194226

In [239]:
trigram_lm.unigram_probability("perro")

0.00020274105911929284

## Probabilidades de 4-gramas y 5-gramas

In [240]:
fivegram_data = NgramData(vocab_max=10000, tokenizer=tokenizer, n=5)

transformed_corpus_5 = fivegram_data.fit(tr_txt)
final_vocab_5 = fivegram_data.final_vocabulary

In [241]:
transformed_corpus_5[0]

['<s>',
 '<s>',
 '<s>',
 '<s>',
 '@usuario',
 '@usuario',
 '@usuario',
 'q',
 'se',
 'puede',
 'esperar',
 'del',
 'maricon',
 'de',
 'closet',
 'de',
 'la',
 'yañez',
 'aun',
 'recuerdo',
 'esa',
 'ves',
 'q',
 'lo',
 'vi',
 'en',
 'zona',
 'rosa',
 'viendo',
 'quien',
 'lo',
 'levantada',
 '</s>']

In [242]:
fivegram_lm = NgramLanguageModel(lambda1=.01, lambda2=.04, lambda3=.15, lambda4=.3, lambda5=.5) # suman 1
fivegram_lm.train(transformed_corpus_5, final_vocab_5)

In [243]:
fivegram_lm.checar_probabilidad()

1.0
0.9999999999999999
0.9999999999999999
1.0
1.0


In [244]:
len(fivegram_lm.fourgram_count), len(fivegram_lm.fivegram_count)

(93199, 95180)

In [245]:
fivegram_lm.fourgram_probability("hola", "mundo", "de", "nlp")

9.997000899730081e-05

In [246]:
fivegram_lm.fourgram_probability("no", "me", "importa", "lo")

0.0006992308460693238

In [247]:
fivegram_lm.fourgram_probability("me", "dan", "ganas", "de")

0.0005995203836930455

In [248]:
fivegram_lm.fourgram_probability("<s>", "<s>", "<s>", "vete")

0.00029184298847220193

In [249]:
fivegram_lm.fivegram_probability("hola", "mundo", "de", "nlp", "en")

9.997000899730081e-05

In [250]:
fivegram_lm.fivegram_probability("no", "me", "importa", "lo", "que")

0.0006993705664901589

In [251]:
fivegram_lm.fivegram_probability("me", "acabo", "de", "dar", "cuenta")

0.0003997601439136518

In [252]:
fivegram_lm.fivegram_probability("<s>", "<s>", "<s>", "<s>", "vete")

0.0003926444604410706

In [253]:
fivegram_lm.probability_of_word_5gram("hola", "mundo", "de", "nlp", "en")

0.0005050280143056712

In [254]:
fivegram_lm.probability_of_word_5gram("no", "me", "importa", "lo", "que")

0.0014812754904915074

In [255]:
fivegram_lm.probability_of_word_5gram("<s>", "<s>", "<s>", "<s>", "vete")

0.00032769214048440274

In [256]:
sum(fivegram_lm.probability_of_word_5gram("no", "me", "importa", "lo", w) for w in fivegram_lm.vocab)

1.0

## Ejemplos con el modelo de 4-gramas

In [257]:
fourgram_data = NgramData(vocab_max=10000, tokenizer=tokenizer, n=4)

transformed_corpus_4 = fourgram_data.fit(tr_txt)
final_vocab_4 = fourgram_data.final_vocabulary

In [258]:
fourgram_lm = NgramLanguageModel(lambda1=.01, lambda2=.09, lambda3=.3, lambda4=.6) # suman 1
fourgram_lm.train(transformed_corpus_4, final_vocab_4)

In [259]:
fourgram_lm.rank_permutations_4gram(["no", "me", "importa", "lo", "que", "digan"])

720


([('digan importa lo que no me', 4.838329381030811e-09),
  ('importa digan lo que no me', 4.007514014074431e-09),
  ('digan me importa lo que no', 1.738160631970915e-09),
  ('no me importa lo que digan', 1.2895663340399608e-09),
  ('digan lo que no me importa', 1.1506692617685534e-09)],
 [('que no me digan lo importa', 1.7957810319831745e-12),
  ('no me que importa digan lo', 1.6542266596938877e-12),
  ('me no que importa digan lo', 1.6540502724589944e-12),
  ('que me no digan lo importa', 1.5672859141120808e-12),
  ('me que no digan lo importa', 1.5578926066351491e-12)])

In [260]:
fourgram_lm.rank_permutations_4gram(["hijo", "de", "tu", "re", "fregada", "madre"])

720


([('re fregada hijo de tu madre', 9.79967597395548e-10),
  ('fregada re hijo de tu madre', 9.79967597395548e-10),
  ('madre fregada hijo de tu re', 2.4738236232283834e-10),
  ('fregada madre hijo de tu re', 2.4738236232283834e-10),
  ('re hijo fregada de tu madre', 1.448627997341385e-10)],
 [('tu madre de re fregada hijo', 1.0438480462989618e-12),
  ('madre tu de fregada hijo re', 1.0437574704208985e-12),
  ('tu madre de fregada re hijo', 1.0434791427082422e-12),
  ('tu madre de re hijo fregada', 1.043075178578627e-12),
  ('tu madre de fregada hijo re', 1.04289221828093e-12)])

In [261]:
fourgram_lm.rank_permutations_4gram(["me", "acabo", "de", "dar", "cuenta"])

120


([('cuenta me acabo de dar', 3.264574310563287e-07),
  ('me acabo de dar cuenta', 1.6116094004885293e-07),
  ('cuenta dar me acabo de', 1.4421448892845687e-07),
  ('dar cuenta me acabo de', 1.4421448892845687e-07),
  ('me cuenta acabo de dar', 1.1399747580196529e-07)],
 [('dar me de acabo cuenta', 1.0172757040412519e-08),
  ('cuenta de me dar acabo', 1.011140497852439e-08),
  ('de cuenta me dar acabo', 1.011140497852439e-08),
  ('cuenta me de acabo dar', 1.0011573934585262e-08),
  ('me cuenta de acabo dar', 1.0011270136549891e-08)])

In [262]:
fourgram_lm.rank_permutations_4gram(["estamos", "en", "clase", "en", "la", "universidad"])

360


([('universidad clase estamos en en la', 5.366991843683034e-11),
  ('clase universidad estamos en en la', 5.366991843683034e-11),
  ('universidad clase estamos en la en', 5.332174160256023e-11),
  ('clase universidad estamos en la en', 5.332174160256023e-11),
  ('estamos clase universidad en en la', 5.120171699879579e-11)],
 [('en la en universidad estamos clase', 9.978047824698773e-13),
  ('la en en estamos clase universidad', 9.977931429326928e-13),
  ('en la en estamos clase universidad', 9.977931429326928e-13),
  ('en en la estamos universidad clase', 9.858787016150571e-13),
  ('en en la estamos clase universidad', 9.85764484235673e-13)])

In [263]:
fourgram_lm.rank_permutations_4gram(["que", "te", "valga", "lo", "que", "piensan", "los"])

2520


([('valga piensan lo que los que te', 4.264097777260943e-13),
  ('piensan valga lo que los que te', 4.264097777260943e-13),
  ('piensan valga los que lo que te', 3.2586830663411103e-13),
  ('valga piensan los que lo que te', 3.258668036276695e-13),
  ('piensan valga lo que te que los', 2.0840174850293542e-13)],
 [('que que los valga te piensan lo', 2.100380800760086e-16),
  ('que que los lo te piensan valga', 2.098159781934289e-16),
  ('que que los te piensan lo valga', 2.0949560014270647e-16),
  ('que que los lo valga te piensan', 2.0947862463518918e-16),
  ('que que los valga lo te piensan', 2.093325328254634e-16)])

In [264]:
fourgram_lm.top_next_words_4gram("no", "me", "importa")

[('<s>', 0.001330324174417399),
 ('lo', 0.0008040864585658335),
 ('</s>', 0.0005363377412464836),
 ('<unk>', 0.00044920008650710126),
 ('que', 0.0003408850354619756)]

In [265]:
fourgram_lm.top_next_words_4gram("me", "dan", "ganas")

[('<s>', 0.0013303291142586595),
 ('de', 0.0011682810411812155),
 ('</s>', 0.0005094126990410653),
 ('que', 0.00035879329736410723),
 (',', 0.00034245369966773076)]

In [266]:
fourgram_lm.top_next_words_4gram("de", "la", "vista")

[('gorda', 0.0016061186260853829),
 ('<s>', 0.0013302697286542422),
 ('</s>', 0.0005093533134366481),
 (',', 0.0003424246889610216),
 ('<unk>', 0.00034129719622508865)]

In [267]:
fourgram_lm.top_next_words_4gram("ya", "me", "tienen")

[('<s>', 0.0013302855433357877),
 ('hasta', 0.0006835150570195801),
 ('</s>', 0.0005182861993556848),
 ('que', 0.00047781530237848215),
 (',', 0.00036020968142689723)]

In [268]:
fourgram_lm.top_next_words_4gram("la", "costumbre", "de")

[('la', 0.00225243581236033),
 ('<unk>', 0.0014937665437240613),
 ('<s>', 0.0013282762220644924),
 ('que', 0.000785504258798158),
 ('mi', 0.000695483965220414)]

In [269]:
fourgram_lm.sequence_probability_4gram(["no", "me", "importa", "lo", "que", "digan"])

1.2895663340399608e-09

In [270]:
fourgram_lm.sequence_probability_4gram(["te", "voy", "a", "partir", "tu", "madre"])

2.5253358235989575e-11

In [271]:
fourgram_lm.sequence_probability_4gram(["me", "acabo", "de", "dar", "cuenta"])

1.6116094004885293e-07

In [272]:
fourgram_lm.sequence_probability_4gram(["estamos", "en", "clase", "en", "la", "universidad"])

4.3250099919493457e-11

In [273]:
fourgram_lm.sequence_probability_4gram(["asi", "te", "queria", "agarrar", "hijo", "de", "la"])

2.3356159245335143e-14

In [274]:
fourgram_lm.generate_from_tokens_as_chatgpt_4gram("no", "me", "importa")

no me importa 

['no', 'me', 'importa']

In [275]:
fourgram_lm.generate_from_tokens_as_chatgpt_4gram("te", "voy", "a")

te voy a <unk> , pero no me <unk> 

['te', 'voy', 'a', '<unk>', ',', 'pero', 'no', 'me', '<unk>']

In [276]:
fourgram_lm.generate_from_tokens_as_chatgpt_4gram("voy", "a", "volver")

voy a volver loca , pero que no me la madre . 

['voy',
 'a',
 'volver',
 'loca',
 ',',
 'pero',
 'que',
 'no',
 'me',
 'la',
 'madre',
 '.']

In [277]:
fourgram_lm.generate_from_tokens_as_chatgpt_4gram("hijo", "de", "tu")

hijo de tu puta madre 

['hijo', 'de', 'tu', 'puta', 'madre']

In [278]:
fourgram_lm.generate_from_tokens_as_chatgpt_4gram("no", "me", "estes")

no me estes <unk> , no se <unk> <unk> 

['no', 'me', 'estes', '<unk>', ',', 'no', 'se', '<unk>', '<unk>']

## Ejemplos con el modelo de 5-gramas

In [279]:
fivegram_lm.rank_permutations_5gram(["no", "me", "importa", "lo", "que", "digan"])

720


([('digan me importa lo que no', 1.1040237853931636e-06),
  ('no me importa lo que digan', 1.0617711719069102e-06),
  ('me digan importa lo que no', 9.080564970793333e-07),
  ('digan no importa lo que me', 8.686161489864774e-07),
  ('no digan importa lo que me', 8.686161489864774e-07)],
 [('no me que lo digan importa', 1.0185598580195637e-08),
  ('que me no lo importa digan', 1.0184107498021986e-08),
  ('que me no lo digan importa', 1.0183947932028576e-08),
  ('me que no lo importa digan', 1.018228535654169e-08),
  ('me que no lo digan importa', 1.0182136591865428e-08)])

In [280]:
fivegram_lm.rank_permutations_5gram(["a", "si", "hijo", "de", "la", "fregada"])

720


([('fregada si a hijo de la', 7.146209690071038e-07),
  ('a fregada si hijo de la', 7.146209690071038e-07),
  ('si a fregada hijo de la', 7.146209690071038e-07),
  ('si fregada a hijo de la', 7.146209690071038e-07),
  ('a si fregada hijo de la', 7.146209690071038e-07)],
 [('si a la de fregada hijo', 1.0421636466114658e-08),
  ('si a de la hijo fregada', 1.0404170574683666e-08),
  ('a si de la hijo fregada', 1.0404170574683666e-08),
  ('si de a la hijo fregada', 1.0400517223606603e-08),
  ('de si a la hijo fregada', 1.0400219059670244e-08)])

In [281]:
fivegram_lm.rank_permutations_5gram(["me", "acabo", "de", "dar", "cuenta"])

120


([('cuenta dar me acabo de', 0.0005557556756877083),
  ('dar cuenta me acabo de', 0.0005557556756877083),
  ('cuenta me dar acabo de', 0.0004059653985493377),
  ('me cuenta dar acabo de', 0.0004059653985493377),
  ('dar me cuenta acabo de', 0.0004059653985493377)],
 [('me cuenta dar de acabo', 9.981848069579523e-05),
  ('dar cuenta me de acabo', 9.98109889385261e-05),
  ('cuenta dar me de acabo', 9.98109889385261e-05),
  ('dar me cuenta de acabo', 9.980949148527489e-05),
  ('me dar cuenta de acabo', 9.980949148527489e-05)])

In [282]:
fivegram_lm.rank_permutations_5gram(["estamos", "en", "clase", "en", "la", "universidad"])

360


([('universidad en clase estamos en la', 1.5061227264268371e-07),
  ('en universidad clase estamos en la', 1.5061227264268371e-07),
  ('en clase universidad estamos en la', 1.5061227264268371e-07),
  ('clase universidad en estamos en la', 1.5061227264268371e-07),
  ('clase en universidad estamos en la', 1.5061227264268371e-07)],
 [('la en estamos en universidad clase', 9.95477825145684e-09),
  ('en la estamos en universidad clase', 9.95477825145684e-09),
  ('en en clase la estamos universidad', 9.935988500294468e-09),
  ('en clase en la estamos universidad', 9.913948338420856e-09),
  ('clase en en la estamos universidad', 9.913948338420856e-09)])

In [283]:
fivegram_lm.rank_permutations_5gram(["no", "me", "estes", "fregando", "hijo", "de", "la"])

5040


([('hijo estes no me fregando de la', 9.220339623533698e-10),
  ('estes hijo no me fregando de la', 9.220339623533698e-10),
  ('no hijo estes me fregando de la', 8.072351452941107e-10),
  ('hijo no estes me fregando de la', 8.072351452941107e-10),
  ('no estes hijo me fregando de la', 8.072332065239207e-10)],
 [('no la de fregando estes me hijo', 2.1613898621155156e-12),
  ('la no de fregando estes me hijo', 2.1613898621155156e-12),
  ('la no fregando de estes me hijo', 2.161375742469198e-12),
  ('no fregando de la estes me hijo', 2.1612532544833332e-12),
  ('no la fregando de estes me hijo', 2.160058366296029e-12)])

In [284]:
fivegram_lm.top_next_words_5gram("no", "me", "importa", "lo")

[('<s>', 0.0016758551506037443),
 ('que', 0.0014812754904915074),
 ('</s>', 0.0004930305349433075),
 ('<unk>', 0.000399557031434093),
 ('de', 0.0003600852380020395)]

In [285]:
fivegram_lm.top_next_words_5gram("que", "traes", "hijo", "de")

[('<s>', 0.0016750985816661385),
 ('la', 0.0011917704486138212),
 ('<unk>', 0.0008588327318388745),
 ('tu', 0.0006996482410338565),
 ('que', 0.0005289144145497536)]

In [286]:
fivegram_lm.top_next_words_5gram("me", "acabo", "de", "dar")

[('<s>', 0.001676100673572861),
 ('</s>', 0.0004932760579124243),
 ('cuenta', 0.00039966512089715356),
 ('<unk>', 0.0003438053093591533),
 ('que', 0.00033139776170576984)]

In [287]:
fivegram_lm.top_next_words_5gram("que", "no", "se", "pierda")

[('<s>', 0.0016761030404373466),
 ('la', 0.0005842727750739748),
 ('</s>', 0.00049327842477691),
 ('<unk>', 0.0003318483379736889),
 ('que', 0.0003314001285702556)]

In [288]:
fivegram_lm.top_next_words_5gram("me", "voy", "a", "volver")

[('<s>', 0.0016760765930977208),
 ('loca', 0.0007590256097526018),
 ('</s>', 0.0004932519774372841),
 ('<unk>', 0.00033182189063406297),
 ('que', 0.0003313736812306297)]

In [289]:
fivegram_lm.sequence_probability_5gram(["no", "me", "importa", "lo", "que", "digan"])

1.0617711719069102e-06

In [290]:
fivegram_lm.sequence_probability_5gram(["no", "me", "estes", "fregando", "hijo", "de"])

6.629740034534107e-08

In [291]:
fivegram_lm.sequence_probability_5gram(["me", "acabo", "de", "dar", "cuenta"])

0.00039966512089715373

In [292]:
fivegram_lm.sequence_probability_5gram(["estamos", "en", "clase", "en", "la", "universidad"])

1.0204933176801041e-07

In [293]:
fivegram_lm.sequence_probability_5gram(["le", "vas", "al", "america", "hijo", "de", "tu"])

4.4371097077448634e-11

In [294]:
fivegram_lm.generate_from_tokens_as_chatgpt_5gram("no", "me", "importa", "lo")

no me importa lo 

['no', 'me', 'importa', 'lo']

In [295]:
fivegram_lm.generate_from_tokens_as_chatgpt_5gram("vete", "mucho", "a", "la")

vete mucho a la verga 

['vete', 'mucho', 'a', 'la', 'verga']

In [296]:
fivegram_lm.generate_from_tokens_as_chatgpt_5gram("no", "me", "estes", "chingando")

no me estes chingando <unk> <unk> <unk> <unk> <unk> <unk> <unk> 

['no',
 'me',
 'estes',
 'chingando',
 '<unk>',
 '<unk>',
 '<unk>',
 '<unk>',
 '<unk>',
 '<unk>',
 '<unk>']

In [297]:
fivegram_lm.generate_from_tokens_as_chatgpt_5gram("me", "voy", "a", "volver")

me voy a volver 

['me', 'voy', 'a', 'volver']

In [298]:
fivegram_lm.generate_from_tokens_as_chatgpt_5gram("<s>", "<s>", "<s>", "<s>")

<s> <s> <s> <s> @usuario @usuario @usuario @usuario 

['<s>', '<s>', '<s>', '<s>', '@usuario', '@usuario', '@usuario', '@usuario']